# Clasificación de Frutas: Entrenamiento con ResNet18

Este notebook documenta paso a paso el proceso completo para desarrollar un clasificador de imágenes de frutas frescas y podridas utilizando aprendizaje por transferencia con ResNet18.

## Objetivo
Crear un modelo de clasificación de imágenes capaz de distinguir entre frutas frescas y podridas, con potencial aplicación en sistemas de control de calidad automatizados para la industria alimentaria.

## Dataset
Utilizaremos el dataset "Fruits Fresh and Rotten for Classification" disponible en Kaggle, que contiene imágenes de 6 tipos de frutas (manzanas, plátanos, naranjas, etc.) en dos estados: fresco y podrido.

## Metodología
1. Transfer Learning con ResNet18 pre-entrenado en ImageNet
2. Fine-tuning solo en la capa fully-connected final
3. Entrenamiento durante 10 épocas con validación
4. Evaluación del rendimiento

## Requisitos
- Python 3.7+
- PyTorch 1.8+
- Torchvision
- KaggleHub (para descargar el dataset)
- Matplotlib (para visualización)

## Paso 1: Importación de Librerías

Importamos todas las librerías necesarias para el proyecto:
- KaggleHub: Para descargar el dataset
- PyTorch: Framework principal para deep learning
- Torchvision: Para modelos pre-entrenados y transformaciones de imágenes
- Matplotlib: Para visualización de resultados
- OS: Para manejo de rutas de archivos

In [ ]:
import kagglehub
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os
from google.colab import drive

print("✅ Librerías importadas correctamente")

## Paso 2: Descarga del Dataset desde Kaggle Hub

Descargamos el dataset directamente desde Kaggle Hub utilizando la API de kagglehub.



In [ ]:
try:
    path = kagglehub.dataset_download("sriramr/fruits-fresh-and-rotten-for-classification")
    print("✅ Dataset descargado correctamente")
    print("Path to dataset files:", path)
except Exception as e:
    print(f"❌ Error al descargar el dataset: {e}")

## Paso 3: Configuración Inicial y Transformaciones

Configuramos los parámetros básicos del modelo y definimos las transformaciones para el preprocesamiento de imágenes:

- **Tamaño del batch**: 8 (ajustable según memoria GPU)
- **Tamaño de imagen**: 224x224 (requerido por ResNet)
- **Transformaciones**:
  - Redimensionamiento
  - Conversión a tensor
  - Normalización (usando medias y std de ImageNet)

**Nota:** La normalización usa valores estándar de ImageNet ya que ResNet fue pre-entrenado con estos valores.

In [ ]:
# Configuración básica
batch_size = 8  # Puede aumentarse si hay suficiente memoria GPU
img_size = 224  # Tamaño requerido por ResNet
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Dispositivo de ejecución: {device}")

# Transformaciones para preprocesamiento de imágenes
transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),  # Redimensionar a 224x224
    transforms.ToTensor(),  # Convertir a tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # Normalizar con valores de ImageNet
                         std=[0.229, 0.224, 0.225])
])

print("✅ Configuración y transformaciones definidas")

## Paso 4: Carga y División del Dataset

Cargamos el dataset completo y lo dividimos en conjuntos de entrenamiento (80%) y validación (20%).

**Detalles importantes:**
- Usamos ImageFolder que automáticamente organiza las imágenes por clases según la estructura de directorios
- La semilla aleatoria (seed=42) asegura reproducibilidad
- Los DataLoaders manejan el cargado eficiente de datos durante el entrenamiento

In [ ]:
try:
    # Cargar dataset completo
    full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)
    class_names = full_dataset.classes
    print(f"🔍 Clases detectadas: {class_names}")
    print(f"📊 Total de imágenes: {len(full_dataset)}")

    # Dividir en entrenamiento y validación
    val_split = 0.2  # 20% para validación
    val_size = int(len(full_dataset) * val_split)
    train_size = len(full_dataset) - val_size
    
    # Fijar semilla para reproducibilidad
    torch.manual_seed(42)
    
    train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])
    print(f"🚂 Imágenes de entrenamiento: {len(train_dataset)}")
    print(f"🧪 Imágenes de validación: {len(val_dataset)}")

    # Crear DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    
    print("✅ Dataset cargado y dividido correctamente")
except Exception as e:
    print(f"❌ Error al cargar el dataset: {e}")

## Paso 5: Definición del Modelo

Cargamos ResNet18 pre-entrenado en ImageNet y adaptamos para nuestro problema:

1. Cargamos el modelo pre-entrenado
2. Congelamos todos los parámetros (transfer learning)
3. Reemplazamos la última capa fully-connected para que tenga la salida adecuada para nuestras clases
4. Movemos el modelo al dispositivo (GPU/CPU)

**Estrategia de Transfer Learning:**
- Congelamos todas las capas excepto la última
- Esto permite reutilizar las características aprendidas en ImageNet
- Solo entrenamos la nueva capa fully-connected para nuestra tarea específica

In [ ]:
try:
    # Cargar ResNet18 pre-entrenado
    model = models.resnet18(pretrained=True)
    print("🎯 ResNet18 cargado (pre-entrenado en ImageNet)")

    # Congelar todos los parámetros
    for param in model.parameters():
        param.requires_grad = False
    print("❄️ Parámetros congelados (excepto última capa)")

    # Reemplazar la capa fully-connected
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, len(class_names))
    print(f"🔄 Capa FC reemplazada para {len(class_names)} clases")

    # Mover modelo al dispositivo
    model = model.to(device)
    print(f"📌 Modelo movido a {device}")
    
    print("✅ Modelo definido correctamente")
except Exception as e:
    print(f"❌ Error al configurar el modelo: {e}")

## Paso 6: Configuración del Entrenamiento

Definimos los parámetros para el entrenamiento:

- **Función de pérdida**: CrossEntropyLoss (adecuada para clasificación)
- **Optimizador**: Adam con learning rate de 0.001
- **Épocas**: 10 (suficiente para fine-tuning)

**Notas:**
- Adam es una buena opción por su adaptabilidad
- El LR es bajo porque solo estamos ajustando la última capa
- Podría implementarse learning rate scheduler para mejor ajuste

In [ ]:
# Función de pérdida (clasificación multi-clase)
criterion = nn.CrossEntropyLoss()

# Optimizador (solo para los parámetros de la última capa)
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# Número de épocas
epochs = 10

print("⚙️ Configuración de entrenamiento:")
print(f"- Función de pérdida: {criterion.__class__.__name__}")
print(f"- Optimizador: {optimizer.__class__.__name__} (lr=0.001)")
print(f"- Épocas: {epochs}")
print("✅ Configuración de entrenamiento completada")

## Paso 7: Entrenamiento del Modelo

Ciclo completo de entrenamiento y validación:

1. **Modo entrenamiento**:
   - Forward pass
   - Cálculo de pérdida
   - Backpropagation
   - Actualización de pesos

2. **Modo evaluación**:
   - Validación sin gradientes
   - Cálculo de pérdida y precisión
   - Monitoreo del rendimiento

**Métricas guardadas:**
- Pérdida de entrenamiento por época
- Pérdida de validación por época
- Precisión en validación

In [ ]:
# Listas para guardar métricas
train_losses = []
val_losses = []
val_accuracies = []

print("🚀 Comenzando entrenamiento...")

for epoch in range(epochs):
    # --- Fase de Entrenamiento ---
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        # Mover datos al dispositivo
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass y optimización
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()

    # Calcular pérdida promedio por época
    epoch_loss = running_loss / len(train_loader)
    train_losses.append(epoch_loss)

    # --- Fase de Validación ---
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            # Calcular precisión
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    # Calcular métricas de validación
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    
    accuracy = 100 * correct / total
    val_accuracies.append(accuracy)
    
    # Mostrar progreso
    print(f"✅ Epoch {epoch+1}/{epochs} - "
          f"Train Loss: {epoch_loss:.4f} - "
          f"Val Loss: {val_loss:.4f} - "
          f"Accuracy: {accuracy:.2f}%")

print("🎉 Entrenamiento completado!")

## Paso 8: Visualización de Resultados

Graficamos las curvas de pérdida y precisión para analizar el rendimiento del modelo:

1. **Curva de pérdida**: Muestra la evolución de la pérdida en entrenamiento y validación
2. **Curva de precisión**: Muestra la precisión en el conjunto de validación por época

**Análisis esperado:**
- La pérdida debería disminuir en ambas curvas
- La precisión debería aumentar
- Si hay overfitting, la pérdida de validación comenzaría a aumentar

In [ ]:
plt.figure(figsize=(12, 5))

# Gráfico de pérdidas
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Entrenamiento')
plt.plot(val_losses, label='Validación')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.title('Curva de Pérdida')
plt.legend()

# Gráfico de precisión
plt.subplot(1, 2, 2)
plt.plot(val_accuracies, label='Validación', color='green')
plt.xlabel('Épocas')
plt.ylabel('Precisión (%)')
plt.title('Precisión en Validación')
plt.legend()

plt.tight_layout()
plt.show()

## Paso 9: Guardado del Modelo

Guardamos el modelo entrenado para su uso futuro:

- Solo guardamos los parámetros del modelo (state_dict)
- Esto permite cargar el modelo en cualquier momento
- El archivo tiene extensión .pth (convención PyTorch)

**Nota:** Para cargar el modelo después, necesitaremos reconstruir la misma arquitectura y luego cargar los parámetros.

In [ ]:
try:
    model_path = "fruitscan_model.pth"
    torch.save(model.state_dict(), model_path)
    print(f"\n💾 Modelo guardado como: {model_path}")
    print(f"Tamaño del archivo: {os.path.getsize(model_path)/1024:.2f} KB")
except Exception as e:
    print(f"❌ Error al guardar el modelo: {e}")